# Notebook 3 — Random Stratified Train / Validation / Test Split

## Goal

Split the labeled dataset into training, validation, and test sets using a random stratified split.

We will keep the same class ratio across the three sets as much as possible, then check the dates and label distribution in each split.

The split will be created before the detailed EDA in Notebook 4.

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

## 1. Load the labeled dataset

The labeled dataset was created in Notebook 2.

We use this artifact as the input for the split instead of rebuilding the data again.

In [2]:
labeled_orders1 = pd.read_csv("../artifacts/labeled_orders1.csv")

print("Shape:", labeled_orders1.shape)
labeled_orders1.head()

Shape: (96470, 28)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,number_of_product_categories,customer_seller_distance_km,customer_seller_same_state,number_of_payments,total_payment,average_installments,max_installments,main_payment_type,delivery_delay_days,late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,1.0,18.576109,True,3.0,38.71,1.0,1.0,voucher,-7.107488,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,1.0,851.495057,False,1.0,141.46,1.0,1.0,boleto,-5.355729,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1.0,514.410611,False,1.0,179.12,3.0,3.0,credit_card,-17.245498,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,1.0,1822.226338,False,1.0,72.20,1.0,1.0,credit_card,-12.980069,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,1.0,29.676608,True,1.0,28.62,1.0,1.0,credit_card,-9.238171,0


In [3]:
print("Columns:")
print(labeled_orders1.columns.tolist())

print("\nTarget distribution:")
print(
    labeled_orders1["late_delivery"]
    .value_counts()
    .sort_index()
)

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'number_of_items', 'number_of_sellers', 'total_price', 'total_freight', 'total_product_weight', 'total_product_volume', 'number_of_product_categories', 'customer_seller_distance_km', 'customer_seller_same_state', 'number_of_payments', 'total_payment', 'average_installments', 'max_installments', 'main_payment_type', 'delivery_delay_days', 'late_delivery']

Target distribution:
late_delivery
0    88644
1     7826
Name: count, dtype: int64


## 2. Check the purchase dates

The purchase date helps us understand the time range of the dataset.

We are not using the date to create the split because we chose a random split, but we still check it before splitting.

In [4]:
labeled_orders1["order_purchase_timestamp"] = pd.to_datetime(
    labeled_orders1["order_purchase_timestamp"],
    errors="coerce"
)

print(
    "Earliest order:",
    labeled_orders1["order_purchase_timestamp"].min()
)

print(
    "Latest order:",
    labeled_orders1["order_purchase_timestamp"].max()
)

Earliest order: 2016-09-15 12:16:38
Latest order: 2018-08-29 15:00:37


## 3. Choose the Splitting Strategy

A random split is used for this version of the pipeline.

Stratification is important because late deliveries are much less common
than on-time deliveries. It keeps the target ratio similar across the
training, validation, and test sets.

The test set will not be used for decisions during model development.

In [5]:
print("Total orders:", len(labeled_orders1))

print(
    "Late delivery ratio:",
    round(labeled_orders1["late_delivery"].mean() * 100, 2),
    "%"
)

Total orders: 96470
Late delivery ratio: 8.11 %


## 4. Create the Training Set

We use 70% of the data for training.

The remaining 30% will be split into validation and test sets.

The target is used for stratification to keep the class distribution similar.

In [6]:
train, temp = train_test_split(
    labeled_orders1,
    test_size=0.30,
    random_state=42,
    stratify=labeled_orders1["late_delivery"]
)

print("Train shape:", train.shape)
print("Temporary shape:", temp.shape)

Train shape: (67529, 28)
Temporary shape: (28941, 28)


## 5. Create Validation and Test Sets

The remaining 30% is divided equally into validation and test sets.

Stratification is used again to preserve the target distribution.

In [7]:
validation, test = train_test_split(
    temp,
    test_size=0.50,
    random_state=42,
    stratify=temp["late_delivery"]
)

print("Validation shape:", validation.shape)
print("Test shape:", test.shape)

Validation shape: (14470, 28)
Test shape: (14471, 28)


In [8]:
print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

print(
    "Total:",
    len(train) + len(validation) + len(test)
)

Train: 67529
Validation: 14470
Test: 14471
Total: 96470


## 6. Check the Label Distribution

We compare the number of on-time and late orders in each split.

The proportions should be very similar because stratification was used.

In [9]:
print("Train label counts:")
print(
    train["late_delivery"]
    .value_counts()
    .sort_index()
)

print("\nValidation label counts:")
print(
    validation["late_delivery"]
    .value_counts()
    .sort_index()
)

print("\nTest label counts:")
print(
    test["late_delivery"]
    .value_counts()
    .sort_index()
)

Train label counts:
late_delivery
0    62051
1     5478
Name: count, dtype: int64

Validation label counts:
late_delivery
0    13296
1     1174
Name: count, dtype: int64

Test label counts:
late_delivery
0    13297
1     1174
Name: count, dtype: int64


In [10]:
print("Train label percentages:")
print(
    (
        train["late_delivery"]
        .value_counts(normalize=True)
        * 100
    )
    .sort_index()
    .round(2)
)

print("\nValidation label percentages:")
print(
    (
        validation["late_delivery"]
        .value_counts(normalize=True)
        * 100
    )
    .sort_index()
    .round(2)
)

print("\nTest label percentages:")
print(
    (
        test["late_delivery"]
        .value_counts(normalize=True)
        * 100
    )
    .sort_index()
    .round(2)
)

Train label percentages:
late_delivery
0    91.89
1     8.11
Name: proportion, dtype: float64

Validation label percentages:
late_delivery
0    91.89
1     8.11
Name: proportion, dtype: float64

Test label percentages:
late_delivery
0    91.89
1     8.11
Name: proportion, dtype: float64


## 7. Compare the Class Ratios

We compare the late-delivery percentage in the full dataset and in each split.

The values should be very close to each other.

In [11]:
distribution = pd.DataFrame({
    "dataset": ["Full", "Train", "Validation", "Test"],
    "late_delivery_ratio": [
        labeled_orders1["late_delivery"].mean(),
        train["late_delivery"].mean(),
        validation["late_delivery"].mean(),
        test["late_delivery"].mean()
    ]
})

distribution["late_delivery_ratio"] = (
    distribution["late_delivery_ratio"] * 100
).round(2)

distribution

,dataset,late_delivery_ratio
0,Full,8.11
1,Train,8.11
2,Validation,8.11
3,Test,8.11


## 8. Check the Date Range in Each Split

Because this is a random split, the date ranges are expected to overlap.

This confirms that the three datasets contain orders from the same general
time period.

In [12]:
for name, dataset in [
    ("Train", train),
    ("Validation", validation),
    ("Test", test)
]:
    print(f"\n{name}")
    print(
        "Earliest:",
        dataset["order_purchase_timestamp"].min()
    )
    print(
        "Latest:",
        dataset["order_purchase_timestamp"].max()
    )


Train
Earliest: 2016-10-03 09:44:50
Latest: 2018-08-29 15:00:37

Validation
Earliest: 2016-09-15 12:16:38
Latest: 2018-08-28 19:27:43

Test
Earliest: 2016-10-03 22:06:03
Latest: 2018-08-28 21:56:12


## 9. Check the Split Sizes

We verify that the final datasets contain approximately:

- 70% training data
- 15% validation data
- 15% test data

In [13]:
total_rows = len(labeled_orders1)

print(
    "Train percentage:",
    round(len(train) / total_rows * 100, 2)
)

print(
    "Validation percentage:",
    round(len(validation) / total_rows * 100, 2)
)

print(
    "Test percentage:",
    round(len(test) / total_rows * 100, 2)
)

Train percentage: 70.0
Validation percentage: 15.0
Test percentage: 15.0


## 10. Check for Overlapping Orders

Each order must belong to only one split.

We check the `order_id` values to make sure there is no overlap between
training, validation, and test data.

In [14]:
train_ids = set(train["order_id"])
validation_ids = set(validation["order_id"])
test_ids = set(test["order_id"])

print(
    "Train / Validation overlap:",
    len(train_ids & validation_ids)
)

print(
    "Train / Test overlap:",
    len(train_ids & test_ids)
)

print(
    "Validation / Test overlap:",
    len(validation_ids & test_ids)
)

Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0


## 11. Final Split Check

Before saving the data, we check the number of rows, unique orders,
and target classes in each split.

In [15]:
for name, dataset in [
    ("Train", train),
    ("Validation", validation),
    ("Test", test)
]:
    print(f"\n{name}")
    print("Rows:", len(dataset))
    print("Unique orders:", dataset["order_id"].nunique())
    print(
        "Duplicate orders:",
        dataset["order_id"].duplicated().sum()
    )
    print(
        "Classes:",
        sorted(dataset["late_delivery"].unique())
    )


Train
Rows: 67529
Unique orders: 67529
Duplicate orders: 0
Classes: [np.int64(0), np.int64(1)]

Validation
Rows: 14470
Unique orders: 14470
Duplicate orders: 0
Classes: [np.int64(0), np.int64(1)]

Test
Rows: 14471
Unique orders: 14471
Duplicate orders: 0
Classes: [np.int64(0), np.int64(1)]


## 12. Save the Split Artifacts

The random split datasets are saved in a separate folder.

This keeps this split independent from any other split version and allows
the next notebooks to read the correct artifacts directly.

In [16]:
random_split_path = "../artifacts/random_split"

os.makedirs(random_split_path, exist_ok=True)

train_path = f"{random_split_path}/train1.csv"
validation_path = f"{random_split_path}/validation1.csv"
test_path = f"{random_split_path}/test1.csv"

train.to_csv(train_path, index=False)
validation.to_csv(validation_path, index=False)
test.to_csv(test_path, index=False)

print("Train saved to:", train_path)
print("Validation saved to:", validation_path)
print("Test saved to:", test_path)

Train saved to: ../artifacts/random_split/train1.csv
Validation saved to: ../artifacts/random_split/validation1.csv
Test saved to: ../artifacts/random_split/test1.csv


In [17]:
print("Train artifact exists:", os.path.exists(train_path))
print(
    "Validation artifact exists:",
    os.path.exists(validation_path)
)
print("Test artifact exists:", os.path.exists(test_path))

print("\nSaved train rows:", len(train))
print("Saved validation rows:", len(validation))
print("Saved test rows:", len(test))

Train artifact exists: True
Validation artifact exists: True
Test artifact exists: True

Saved train rows: 67529
Saved validation rows: 14470
Saved test rows: 14471


## Final Summary

In this notebook, the labeled dataset was divided into training, validation, and test sets using a random stratified split.

The split preserved the proportion of late and on-time deliveries across the three datasets.

The final split contains:

- Training: 67,529 orders
- Validation: 14,470 orders
- Test: 14,471 orders

The split datasets were saved as artifacts for the following notebooks.